# SEC 10-K Section Extractor (Colab → Google Drive)

Downloads 10-K filings from SEC EDGAR and extracts **Item 1** (Business), **Item 1A** (Risk Factors), and **Item 7** (MD&A) into clean JSON files **saved directly to Google Drive**.

**Architecture:**
- **8 I/O threads** download filings in parallel (network-bound, GIL-free)
- **Batched processing** (40 filings at a time) caps RAM usage
- Each filing is downloaded into memory, parsed, then freed — raw filings never touch disk
- JSONs write directly to your mounted Google Drive as they're extracted
- **No GPU needed** — this is network I/O + regex/HTML parsing (CPU-only)

**Usage:** Edit config in Cell 3, put your CIK CSV on Drive, and Run All.

## 1. Mount Google Drive & install dependencies

In [11]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

!pip install -q cssutils pathos lxml beautifulsoup4 tqdm requests pandas numpy click

import os

# Clone edgar-crawler (contains extract_items.py, item_lists.py, logger.py, __init__.py)
if not os.path.isdir('/content/edgar-crawler'):
    !git clone https://github.com/lefterisloukas/edgar-crawler.git /content/edgar-crawler

os.chdir('/content/edgar-crawler')
print(f'Working directory: {os.getcwd()}')

Mounted at /content/drive
Working directory: /content/edgar-crawler


## 2. Configuration

Edit the settings below. **CIK file** and **output folder** are both on Google Drive.

In [12]:
# ============================================================
# CONFIGURATION — Edit these values
# ============================================================

CONFIG = {
    # Year range
    'start_year': 2004,
    'end_year': 2026,
    'quarters': [1, 2, 3, 4],

    # Filing types
    'filing_types': ['10-K', '10-K405', '10-KT'],

    # Sections to extract (10-K item numbers)
    'items_to_extract': ['1', '1A', '7'],

    # CIK file ON GOOGLE DRIVE
    'cik_file': '/content/drive/MyDrive/data/sp500_historical_ciks.csv',

    # SEC requires a User-Agent with your name and email
    'user_agent': 'Rens Gerritsen 658549rg@eur.nl',

    # Output DIRECTLY TO GOOGLE DRIVE (saves continuously, survives disconnects)
    'output_folder': '/content/drive/MyDrive/FML_project_4',

    # Index cache (local Colab disk — fast, rebuilt if session resets)
    'indices_folder': '/content/edgar-crawler/INDICES',

    # Performance
    'io_workers': 8,       # download threads (SEC allows ~10 req/s)
    'batch_size': 40,      # filings in memory at once (Colab has 12GB+ RAM)
    'remove_tables': True,
    'skip_existing': True, # skip already-extracted JSONs on rerun
}

os.makedirs(CONFIG['output_folder'], exist_ok=True)
os.makedirs(CONFIG['indices_folder'], exist_ok=True)

print('Config ready.')
print(f'  Years:   {CONFIG["start_year"]}-{CONFIG["end_year"]}')
print(f'  Sections: {CONFIG["items_to_extract"]}')
print(f'  CIK file: {CONFIG["cik_file"]}')
print(f'  Output:   {CONFIG["output_folder"]}')
print(f'  Batch:    {CONFIG["batch_size"]} filings at a time')

Config ready.
  Years:   2004-2026
  Sections: ['1', '1A', '7']
  CIK file: /content/drive/MyDrive/data/sp500_historical_ciks.csv
  Output:   /content/drive/MyDrive/FML_project_4
  Batch:    40 filings at a time


In [13]:
# ⚠️ DELETE OLD RESULTS — old JSONs have empty item_1A from the buggy version
# Comment this out after your first successful re-run
import shutil
if os.path.exists(CONFIG['output_folder']):
    count = sum(1 for _ in os.scandir(CONFIG['output_folder']) if _.is_dir())
    print(f"Found {count} CIK folders in {CONFIG['output_folder']}")
    confirm = input('Delete old results and re-extract? (yes/no): ')
    if confirm.strip().lower() == 'yes':
        shutil.rmtree(CONFIG['output_folder'])
        os.makedirs(CONFIG['output_folder'])
        print('Deleted. Will re-extract all filings.')
    else:
        print('Kept old results. Old JSONs with empty item_1A will be SKIPPED (not re-extracted).')
else:
    print('No old results found — starting fresh.')

Found 71 CIK folders in /content/drive/MyDrive/FML_project_4
Kept old results. Old JSONs with empty item_1A will be SKIPPED (not re-extracted).


## 3. Patch ExtractItems with in-memory extraction

In [14]:
import sys
import re
import json
import os
import logging

# Add edgar-crawler to path
sys.path.insert(0, os.getcwd())

# Suppress cssutils warnings
import cssutils
cssutils.log.setLevel(logging.CRITICAL)

from extract_items import ExtractItems
from bs4 import BeautifulSoup

regex_flags = re.IGNORECASE | re.DOTALL | re.MULTILINE


def extract_items_from_content(self, content, filing_metadata):
    """
    Extract items from raw filing content held in memory.
    Same logic as extract_items() but reads from a string, not disk.
    """
    content = re.sub(r'<PDF>.*?</PDF>', '', content, flags=regex_flags)
    documents = re.findall('<DOCUMENT>.*?</DOCUMENT>', content, flags=regex_flags)

    doc_report = None
    found, is_html = False, False

    # Pick the LARGEST matching <DOCUMENT> block — the actual 10-K/10-Q/8-K.
    # Full submission .txt files contain many documents (exhibits like EX-10,
    # EX-13, etc.) that also start with '10'. Without this, we'd overwrite
    # doc_report with a small exhibit and lose Item 1A, 7, etc.
    best_doc = None
    best_doc_len = 0
    for doc in documents:
        doc_type = re.search(r'\n[^\S\r\n]*<TYPE>(.*?)\n', doc, flags=regex_flags)
        doc_type = doc_type.group(1) if doc_type else None
        if doc_type and doc_type.startswith(('10', '8')):
            if len(doc) > best_doc_len:
                best_doc = doc
                best_doc_len = len(doc)

    if best_doc is not None:
        doc_report = BeautifulSoup(best_doc, 'lxml')
        is_html = bool(doc_report.find('td')) and bool(doc_report.find('tr'))
        if not is_html:
            doc_report = best_doc
        found = True

    if not found:
        doc_report = BeautifulSoup(content, 'lxml')
        is_html = bool(doc_report.find('td')) and bool(doc_report.find('tr'))
        if not is_html:
            doc_report = content

    if self.remove_tables:
        doc_report = self.remove_html_tables(doc_report, is_html=is_html)

    doc_report = self.handle_spans(doc_report, is_html=is_html)

    json_content = {
        'cik': filing_metadata['CIK'],
        'company': filing_metadata['Company'],
        'filing_type': filing_metadata['Type'],
        'filing_date': filing_metadata['Date'],
        'period_of_report': filing_metadata.get('Period of Report'),
        'sic': filing_metadata.get('SIC'),
        'state_of_inc': filing_metadata.get('State of Inc'),
        'state_location': filing_metadata.get('State location'),
        'fiscal_year_end': filing_metadata.get('Fiscal Year End'),
        'filing_html_index': filing_metadata.get('html_index'),
        'htm_filing_link': filing_metadata.get('htm_file_link'),
        'complete_text_filing_link': filing_metadata.get('complete_text_file_link'),
    }

    text = ExtractItems.strip_html(str(doc_report))
    text = ExtractItems.clean_text(text)

    if filing_metadata['Type'] == '10-Q':
        part_texts = self.get_10q_parts(text, filing_metadata)

    positions = []
    all_items_null = True
    for i, item_index in enumerate(self.items_list):
        next_item_list = self.items_list[i + 1:]

        if 'part' in item_index:
            if i != 0:
                if self.items_list[i - 1].split('__')[0] != item_index.split('__')[0]:
                    positions = []
            text = part_texts[item_index.split('__')[0]]
            if item_index.split('__')[0] not in json_content:
                parts_text = ExtractItems.remove_multiple_lines(
                    part_texts[item_index.split('__')[0].strip()]
                )
                json_content[item_index.split('__')[0]] = parts_text

        if 'part' in self.items_list[i - 1] and item_index == 'SIGNATURE':
            item_section = part_texts[item_index]
        else:
            item_section, positions = self.parse_item(
                text, item_index, next_item_list, positions
            )

        item_section = ExtractItems.remove_multiple_lines(item_section.strip())

        if item_index in self.items_to_extract:
            if item_section != '':
                all_items_null = False
            if item_index == 'SIGNATURE':
                if self.include_signature:
                    json_content[f'{item_index}'] = item_section
            else:
                if 'part' in item_index:
                    json_content[
                        item_index.split('__')[0] + '_item_' + item_index.split('__')[1]
                    ] = item_section
                else:
                    json_content[f'item_{item_index}'] = item_section

    if all_items_null:
        return None

    return json_content


def process_filing_from_content(self, content, filing_metadata, json_filename):
    """
    Extract items from in-memory content and save only the JSON.
    """
    self.determine_items_to_extract(filing_metadata)

    type_folder = os.path.join(self.extracted_files_folder, filing_metadata['Type'])
    absolute_json_filename = os.path.join(type_folder, json_filename)

    if self.skip_extracted_filings and os.path.exists(absolute_json_filename):
        return 0

    json_content = self.extract_items_from_content(content, filing_metadata)

    os.makedirs(type_folder, exist_ok=True)

    if json_content is not None:
        with open(absolute_json_filename, 'w', encoding='utf-8') as fp:
            json.dump(json_content, fp, indent=4, ensure_ascii=False)

    return 1


# Monkey-patch the methods onto ExtractItems
ExtractItems.extract_items_from_content = extract_items_from_content
ExtractItems.process_filing_from_content = process_filing_from_content

print('ExtractItems patched with in-memory methods.')

ExtractItems patched with in-memory methods.


## 4. Download EDGAR index files

In [15]:
import itertools
import math
import tempfile
import zipfile
import time
from datetime import datetime

import requests
from requests.adapters import HTTPAdapter
from urllib3.util import Retry


def requests_retry_session(retries=5, backoff_factor=0.5,
                           status_forcelist=(400,401,403,500,502,503,504,505),
                           session=None):
    session = session or requests.Session()
    retry = Retry(total=retries, read=retries, connect=retries,
                  backoff_factor=backoff_factor, status_forcelist=status_forcelist)
    adapter = HTTPAdapter(max_retries=retry)
    session.mount('http://', adapter)
    session.mount('https://', adapter)
    return session


indices_folder = CONFIG['indices_folder']
os.makedirs(indices_folder, exist_ok=True)

base_url = 'https://www.sec.gov/Archives/edgar/full-index/'

for year in range(CONFIG['start_year'], CONFIG['end_year'] + 1):
    for quarter in CONFIG['quarters']:
        if year == datetime.now().year and quarter > math.ceil(datetime.now().month / 3):
            break

        fname = f'{year}_QTR{quarter}.tsv'
        fpath = os.path.join(indices_folder, fname)

        if os.path.exists(fpath):
            continue

        url = f'{base_url}/{year}/QTR{quarter}/master.zip'
        try:
            with tempfile.TemporaryFile(mode='w+b') as tmp:
                resp = requests_retry_session(retries=5, backoff_factor=0.2).get(
                    url, headers={'User-agent': CONFIG['user_agent']}
                )
                tmp.write(resp.content)
                with zipfile.ZipFile(tmp).open('master.idx') as f:
                    lines = [line.decode('latin-1') for line in itertools.islice(f, 11, None)]
                    lines = [
                        line.strip() + '|' + line.split('|')[-1].replace('.txt', '-index.html')
                        for line in lines
                    ]
                with open(fpath, 'w+', encoding='utf-8') as out:
                    out.write(''.join(lines))
            print(f'  {fname} downloaded')
        except Exception as e:
            print(f'  {fname} FAILED: {e}')

print('\nIndex download complete.')


Index download complete.


## 5. Filter indices by CIK and filing type

In [16]:
import pandas as pd
import numpy as np

FILINGS_CACHE = os.path.join(CONFIG['output_folder'], '_filings_index.parquet')

if CONFIG['skip_existing'] and os.path.exists(FILINGS_CACHE):
    filings_df = pd.read_parquet(FILINGS_CACHE)
    print(f'Loaded cached filings index: {len(filings_df)} filings')
else:
    # Load CIKs from uploaded CSV
    df_cik = pd.read_csv(CONFIG['cik_file'], header=None)
    cik_list = (
        df_cik.iloc[:, 0].astype(str).str.strip()
        .str.replace(r'\.0$', '', regex=True).tolist()
    )
    if cik_list[0].lower() == 'cik':
        cik_list = cik_list[1:]
    ciks = [str(c).strip().lstrip('0') for c in cik_list if str(c).isdigit()]
    print(f'Loaded {len(ciks)} CIKs')

    # Load and filter TSV indices
    tsv_col_names = [
        'CIK', 'Company', 'Type', 'Date', 'complete_text_file_link',
        'html_index', 'Filing Date', 'Period of Report', 'SIC',
        'htm_file_link', 'State of Inc', 'State location',
        'Fiscal Year End', 'filename',
    ]

    all_dfs = []
    for year in range(CONFIG['start_year'], CONFIG['end_year'] + 1):
        for quarter in CONFIG['quarters']:
            fpath = os.path.join(indices_folder, f'{year}_QTR{quarter}.tsv')
            if not os.path.isfile(fpath):
                continue
            df = pd.read_csv(fpath, sep='|', header=None, dtype=str, names=tsv_col_names)
            df['CIK'] = df['CIK'].astype(str).str.lstrip('0')
            df['complete_text_file_link'] = 'https://www.sec.gov/Archives/' + df['complete_text_file_link']
            df['html_index'] = 'https://www.sec.gov/Archives/' + df['html_index']
            df = df[df['Type'].isin(CONFIG['filing_types'])]
            if ciks:
                df = df[df['CIK'].isin(ciks)]
            all_dfs.append(df)

    filings_df = pd.concat(all_dfs, ignore_index=True)
    filings_df.to_parquet(FILINGS_CACHE, index=False)
    print(f'Built and cached filings index: {len(filings_df)} filings -> {FILINGS_CACHE}')

print(f'\nTotal matched filings: {len(filings_df)}')
filings_df[['CIK', 'Company', 'Type', 'Date']].head(10)

Loaded 1046 CIKs
Built and cached filings index: 16199 filings -> /content/drive/MyDrive/FML_project_4/_filings_index.parquet

Total matched filings: 16199


,CIK,Company,Type,Date
0,1000180,SANDISK CORP,10-K,2004-03-12
1,1000228,SCHEIN HENRY INC,10-K,2004-03-09
2,1000697,WATERS CORP /DE/,10-K,2004-03-12
3,1000736,CAREMARK RX INC,10-K,2004-03-09
4,1001082,ECHOSTAR COMMUNICATIONS CORP,10-K,2004-03-26
5,1001288,LEXMARK INTERNATIONAL INC /KY/,10-K,2004-03-12
6,1002910,AMEREN CORP,10-K,2004-03-09
7,1004155,AGL RESOURCES INC,10-K,2004-02-06
8,1004434,AFFILIATED MANAGERS GROUP INC,10-K,2004-03-15
9,1004440,CONSTELLATION ENERGY GROUP INC,10-K,2004-03-10


## 6. Download & Extract (threaded, batched, saves to Drive)

Each batch: 8 threads download → extract sequentially → free memory → next batch.  
JSONs are written directly to your Google Drive as they complete.  
If Colab disconnects, just **Run All** again — `skip_existing` will resume where you left off.

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.notebook import tqdm
import gc
import math

output_folder = CONFIG['output_folder']


# ---- Helpers for parquet output ----
def parquet_path_for(cik):
    return os.path.join(output_folder, cik, f'{cik}_filings.parquet')


def append_to_parquet(json_content):
    """Append one extracted filing row to the per-CIK parquet file."""
    cik = json_content['cik']
    filing_date = json_content.get('filing_date', '') or ''
    json_content['year'] = int(filing_date[:4]) if len(filing_date) >= 4 else None

    cik_folder = os.path.join(output_folder, cik)
    os.makedirs(cik_folder, exist_ok=True)
    pq_path = parquet_path_for(cik)

    new_row = pd.DataFrame([json_content])
    if os.path.exists(pq_path):
        existing = pd.read_parquet(pq_path)
        combined = pd.concat([existing, new_row], ignore_index=True)
        combined.to_parquet(pq_path, index=False)
    else:
        new_row.to_parquet(pq_path, index=False)


# ---- Download function (runs in I/O thread) ----
def download_filing(series_dict, user_agent, filing_types):
    """
    Download a single filing into memory.
    Returns (content, metadata) or None on failure.
    """
    raw_content = None

    htm_link = str(series_dict.get('htm_file_link', '')).strip()
    if htm_link and htm_link.startswith('http'):
        try:
            resp = requests_retry_session(retries=3, backoff_factor=0.3).get(
                htm_link, headers={'User-agent': user_agent}
            )
            if resp.status_code == 200 and len(resp.text) > 500:
                raw_content = resp.text
        except Exception:
            pass

    txt_link = str(series_dict.get('complete_text_file_link', '')).strip()
    if raw_content is None and txt_link and txt_link.startswith('http'):
        try:
            resp = requests_retry_session(retries=3, backoff_factor=0.3).get(
                txt_link, headers={'User-agent': user_agent}
            )
            if resp.status_code == 200 and len(resp.text) > 500:
                raw_content = resp.text
        except Exception:
            pass

    if raw_content is None:
        html_index = str(series_dict.get('html_index', '')).strip()
        if not html_index or not html_index.startswith('http'):
            return None
        try:
            resp = requests_retry_session(retries=3, backoff_factor=0.3).get(
                html_index, headers={'User-agent': user_agent}
            )
            soup = BeautifulSoup(resp.content, 'lxml')
            filing_url = None
            for table in soup.find_all('table'):
                if table.attrs.get('summary') == 'Document Format Files':
                    for tr in table.find_all('tr')[1:]:
                        tds = tr.find_all('td')
                        if len(tds) >= 4:
                            doc_type = tds[3].get_text(strip=True)
                            link_tag = tds[2].find('a')
                            if doc_type in filing_types and link_tag:
                                href = link_tag.get('href', '')
                                if href:
                                    filing_url = ('https://www.sec.gov' + href
                                                  if href.startswith('/') else href)
                                    break
            if filing_url:
                resp2 = requests_retry_session(retries=3, backoff_factor=0.3).get(
                    filing_url, headers={'User-agent': user_agent}
                )
                if resp2.status_code == 200 and len(resp2.text) > 500:
                    raw_content = resp2.text
                    series_dict['htm_file_link'] = filing_url
        except Exception:
            pass

    if raw_content is None:
        return None

    time.sleep(0.12)  # SEC rate limit
    return (raw_content, series_dict)


# ---- Build set of already-processed links from existing parquets ----
processed_links = set()
if CONFIG['skip_existing']:
    for entry in os.scandir(output_folder):
        if entry.is_dir():
            pq = parquet_path_for(entry.name)
            if os.path.exists(pq):
                try:
                    df_ex = pd.read_parquet(pq, columns=['complete_text_filing_link'])
                    processed_links.update(df_ex['complete_text_filing_link'].dropna().tolist())
                except Exception:
                    pass
    print(f'Already extracted: {len(processed_links)} filings in parquet files')


# ---- Build list of filings to process ----
to_process = []
for i in range(len(filings_df)):
    s = filings_df.iloc[i]
    txt_link = str(s.get('complete_text_file_link', '')).strip()
    if CONFIG['skip_existing'] and txt_link in processed_links:
        continue
    to_process.append(dict(s))

print(f'Filings to process: {len(to_process)} (skipped {len(filings_df) - len(to_process)} already extracted)')

if len(to_process) == 0:
    print('Nothing to do!')
else:
    BATCH_SIZE = CONFIG['batch_size']
    IO_WORKERS = CONFIG['io_workers']

    success = 0
    failed = 0
    skipped = 0

    total_batches = math.ceil(len(to_process) / BATCH_SIZE)
    pbar = tqdm(total=len(to_process), desc='Processing')

    for batch_idx in range(total_batches):
        batch = to_process[batch_idx * BATCH_SIZE : (batch_idx + 1) * BATCH_SIZE]

        # Phase 1: Download batch with parallel I/O threads
        downloaded = []
        with ThreadPoolExecutor(max_workers=IO_WORKERS) as dl_pool:
            futures = [
                dl_pool.submit(
                    download_filing, s, CONFIG['user_agent'], CONFIG['filing_types']
                )
                for s in batch
            ]
            for f in as_completed(futures):
                result = f.result()
                if result is not None:
                    downloaded.append(result)
                else:
                    failed += 1
                    pbar.update(1)

        # Phase 2: Extract sections (CPU-bound, runs in main process)
        extraction = ExtractItems(
            remove_tables=CONFIG['remove_tables'],
            items_to_extract=CONFIG['items_to_extract'],
            include_signature=False,
            raw_files_folder='',
            extracted_files_folder=output_folder,
            skip_extracted_filings=False,  # skip logic handled above
        )
        from item_lists import item_list_10k
        extraction.items_list = item_list_10k

        for raw_content, meta in downloaded:
            try:
                json_content = extraction.extract_items_from_content(raw_content, meta)
                if json_content is not None:
                    append_to_parquet(json_content)
                    success += 1
                else:
                    failed += 1
            except Exception as e:
                failed += 1
            pbar.update(1)

        del downloaded
        del extraction
        gc.collect()

    pbar.close()

    print(f'\n{"=" * 50}')
    print(f'Done!')
    print(f'  Extracted: {success}')
    print(f'  Skipped:   {skipped}')
    print(f'  Failed:    {failed}')
    print(f'  Saved to:  {output_folder}/')
    print(f'\nAll results are already on your Google Drive.')


Already extracted: 0 filings in parquet files
Filings to process: 16199 (skipped 0 already extracted)


Processing:   0%|          | 0/16199 [00:00<?, ?it/s]

## 7. Preview results (already on Drive)

In [ ]:
import glob

pq_files = sorted(glob.glob(f'{output_folder}/**/*_filings.parquet', recursive=True))
print(f'Total parquet files on Drive: {len(pq_files)}')

if pq_files:
    sample_df = pd.read_parquet(pq_files[0])
    sample = sample_df.iloc[0].to_dict()
    print(f'\nSample: {os.path.basename(pq_files[0])} ({len(sample_df)} filings, years {sample_df["year"].min()}–{sample_df["year"].max()})')
    print(f'  CIK:     {sample.get("cik")}')
    print(f'  Company: {sample.get("company")}')
    print(f'  Date:    {sample.get("filing_date")}')
    print(f'  Year:    {sample.get("year")}')
    for key in ['item_1', 'item_1A', 'item_7']:
        val = sample.get(key, '')
        length = len(val) if val else 0
        preview = (val[:150] + '...') if val and len(val) > 150 else (val or '(empty)')
        print(f'  {key}: {length:,} chars — "{preview}"')
    print(f'\nAll filings for this company:')
    print(sample_df[['year', 'filing_type', 'filing_date', 'company']].sort_values('year').to_string(index=False))
